# Lot D · Silver · étape 4 sur Kaggle : 3 graines, ablations et entrées du pipeline (Qwen2.5-7B-Instruct, 4 bits)

Même contenu que `notebooks/lot_d_llm_colab.ipynb`, adapté à Kaggle. Tout se fait sur le **dev** ; le test gelé n'est jamais lu.

**Réglages (panneau de droite › Session options)** : Accelerator **GPU T4 x2** (un seul est utilisé) ; Internet **On**
(nécessite un téléphone vérifié).

**Lancement recommandé** : bouton **Save Version › Save & Run All (Commit)**. Le notebook tourne en arrière-plan
(~1 h 40), même navigateur fermé. À la fin : ouvrir la version › onglet **Output** › télécharger
`sorties_etape4.zip`. Tous les résultats (tableaux, comparaisons, résumé des journaux) sont dans ce zip.

| Section | Sortie | Rôle |
|---|---|---|
| 4 | `llm_e4_dev_s42` | configuration retenue (étape 3c), graine 42 |
| 5 | `llm_e4_dev_s42_pipeline` | **entrées du pipeline** : entités du Lot B + anonymisation B6c `union_seuil` |
| 6 | `llm_e4_dev_s42_sans_entites` | **ablation** : sans guidage par les entités |
| 8 | `llm_e4_dev_s13`, `llm_e4_dev_s2026` | **deux autres graines** |

## 1. Un seul GPU, récupérer la branche `lot-d-silver` (cellule ré-exécutable)

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'  # Kaggle donne 2 T4 : le modèle 4 bits tient sur un seul, plus rapide
%cd /kaggle/working
if os.path.isdir('fardi-caseforge/.git'):
    !git -C fardi-caseforge pull -q origin lot-d-silver
else:
    !git clone -q -b lot-d-silver https://github.com/onsalouini/fardi-caseforge.git
%cd /kaggle/working/fardi-caseforge
!git log --oneline -1
!nvidia-smi -L

## 2. Dépendances

In [ ]:
!pip install -q -U "transformers>=4.44" accelerate bitsandbytes

## 3. Essai rapide : 2 documents du dev
Chaque ligne doit afficher `essais 1` et `repli False`.

In [ ]:
!python generation/llm.py --split dev --graine 42 --limite 2 --out generation/sorties/essai.jsonl

## 4. Configuration retenue, graine 42 (~18 min)

In [ ]:
!python generation/llm.py --split dev --graine 42 --reprendre --out generation/sorties/llm_e4_dev_s42.jsonl

## 5. Entrées du pipeline : entités du Lot B et anonymisation B6c (~18 min)

In [ ]:
!python generation/llm.py --split dev --graine 42 --reprendre --entites reports/predictions/lotb_dev_v2.jsonl --anonymisation results/lotb/b6c_anonymisation_dev_union_seuil.jsonl --out generation/sorties/llm_e4_dev_s42_pipeline.jsonl
!python generation/gabarit.py --split dev --version 2 --entites reports/predictions/lotb_dev_v2.jsonl --anonymisation results/lotb/b6c_anonymisation_dev_union_seuil.jsonl --out generation/sorties/gabarit_v2_dev_pipeline.jsonl

## 6. Ablation : sans guidage par les entités (~18 min)

In [ ]:
!python generation/llm.py --split dev --graine 42 --reprendre --sans-entites --out generation/sorties/llm_e4_dev_s42_sans_entites.jsonl

## 7. Sauvegarde intermédiaire (`sorties_etape4_partiel.zip` dans Output)

In [ ]:
import shutil
from IPython.display import FileLink
shutil.make_archive('/kaggle/working/sorties_etape4_partiel', 'zip', 'generation', 'sorties')
FileLink('../sorties_etape4_partiel.zip')

## 8. Deux autres graines : 13 et 2026 (~36 min)

In [ ]:
for graine in [13, 2026]:
    !python generation/llm.py --split dev --graine {graine} --reprendre --out generation/sorties/llm_e4_dev_s{graine}.jsonl

## 9. Ablations du post-traitement (rejeu des réponses de la graine 42, sans GPU) et gabarit v2

In [ ]:
J = 'generation/sorties/llm_e4_dev_s42.journal.jsonl'
for option, nom in [('--sans-nettoyage', 'sans_nettoyage'), ('--sans-demarche-guidee', 'sans_demarche_guidee'), ('--sans-controle', 'sans_controle')]:
    !python generation/llm.py --split dev --graine 42 {option} --rejouer {J} --out generation/sorties/llm_e4_dev_s42_{nom}.jsonl > /dev/null
!python generation/gabarit.py --split dev --version 2 --out generation/sorties/gabarit_v2_dev.jsonl

## 10. Évaluation et comparaisons (aussi écrites dans `generation/sorties/resume_etape4.txt`)

In [ ]:
import subprocess
RESUME = []
def sortir(*lignes):
    print(*lignes, sep=chr(10))
    RESUME.extend(lignes)
def evaluer(nom):
    sortie = subprocess.run(['python', 'generation/evaluer.py', '--gen', f'generation/sorties/{nom}.jsonl', '--ref', 'data/final/fiches_dev.jsonl',
                             '--docs', 'data/final/documents_dev.jsonl', '--out', f'generation/sorties/eval_{nom}.md'],
                            capture_output=True, text=True).stdout.splitlines()
    sortir('=' * 15 + ' ' + nom, *sortie[5:17])
for nom in ['gabarit_v2_dev', 'llm_e4_dev_s42', 'llm_e4_dev_s13', 'llm_e4_dev_s2026', 'gabarit_v2_dev_pipeline', 'llm_e4_dev_s42_pipeline',
            'llm_e4_dev_s42_sans_entites', 'llm_e4_dev_s42_sans_nettoyage', 'llm_e4_dev_s42_sans_demarche_guidee', 'llm_e4_dev_s42_sans_controle']:
    evaluer(nom)

In [ ]:
S = 'generation/sorties/'
graines = ','.join(S + f'llm_e4_dev_s{g}.jsonl' for g in [13, 42, 2026])
comparaisons = [
    ('1_llm_3graines_vs_gabarit', f'llm={graines}', f'gabarit={S}gabarit_v2_dev.jsonl'),
    ('2_pipeline_llm_vs_gabarit', f'llm_pipeline={S}llm_e4_dev_s42_pipeline.jsonl', f'gabarit_pipeline={S}gabarit_v2_dev_pipeline.jsonl'),
    ('3_llm_reference_vs_pipeline', f'entrees_reference={S}llm_e4_dev_s42.jsonl', f'entrees_pipeline={S}llm_e4_dev_s42_pipeline.jsonl'),
    ('4_avec_vs_sans_entites', f'avec_entites={S}llm_e4_dev_s42.jsonl', f'sans_entites={S}llm_e4_dev_s42_sans_entites.jsonl'),
]
for nom, a, b in comparaisons:
    sortir(subprocess.run(['python', 'generation/comparer.py', '--split', 'dev', '--a', a, '--b', b, '--out', f'{S}comparaison_{nom}.md'],
                          capture_output=True, text=True).stdout)

## 11. Résumé des journaux et zip final (`sorties_etape4.zip` dans Output)

In [ ]:
import json
for nom in ['llm_e4_dev_s42', 'llm_e4_dev_s13', 'llm_e4_dev_s2026', 'llm_e4_dev_s42_pipeline', 'llm_e4_dev_s42_sans_entites']:
    j = [json.loads(l) for l in open(f'generation/sorties/{nom}.journal.jsonl', encoding='utf-8')]
    sortir('=' * 10 + f' {nom} {len(j)} documents')
    for cle in ['retires_non_ancres', 'citations_corrigees', 'retires_sensibles', 'retires_suivi', 'retires_doublons', 'demarche_guidee']:
        sortir(f'{cle:22}: ' + str(sum(len(x.get(cle, [])) for x in j)))
    sortir(f"replis gabarit        : {sum(x['repli_gabarit'] for x in j)} | essais > 1 : {sum(x['essais'] > 1 for x in j)}",
           f"temps moyen / doc (s) : {round(sum(x['temps_s'] for x in j) / len(j), 1)}")
open('generation/sorties/resume_etape4.txt', 'w', encoding='utf-8').write(chr(10).join(RESUME) + chr(10))

In [ ]:
import shutil
from IPython.display import FileLink
shutil.make_archive('/kaggle/working/sorties_etape4', 'zip', 'generation', 'sorties')
FileLink('../sorties_etape4.zip')